# Importing Packages

In [0]:
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, when, to_date, trim, to_timestamp, regexp_replace, lag, sum, date_trunc
from pyspark.sql.window import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, lit
load_dotenv()

# Defining Variables

In [0]:
SILVER_SCHEMA_PATH=os.getenv('SILVER_SCHEMA_PATH')
GOLD_SCHEMA_PATH=os.getenv('GOLD_SCHEMA_PATH')

# Creating Schema

In [0]:
spark.sql(f"""create schema if not exists {GOLD_SCHEMA_PATH}""")

# Creating Fact and Dimension Tables

## dim_customer

In [0]:
customer_df=spark.read.table(f"""{SILVER_SCHEMA_PATH}.`silver_customer`""")

### Applying SCD Type 2 for dim_customer

In [0]:
target_path = f"{GOLD_SCHEMA_PATH}.dim_customer"

new_data = customer_df.withColumn("valid_from", current_timestamp()) \
                      .withColumn("valid_to", lit(None).cast("timestamp")) \
                      .withColumn("is_current", lit(True))

if spark.catalog.tableExists(target_path):
    target_table = DeltaTable.forName(spark, target_path)
    
    target_table.alias("t").merge(
        new_data.alias("s"),
        "t.customer_id = s.customer_id AND t.is_current = true"
    ).whenMatchedUpdate(
        set = {"is_current": "false", "valid_to": "current_timestamp()"}
    ).execute()
    
    new_data.write.format("delta").mode("append").saveAsTable(target_path)
else:
    new_data.write.format("delta").saveAsTable(target_path)

## dim_product

In [0]:
product_df=spark.read.table(f"""{SILVER_SCHEMA_PATH}.`silver_product`""")

## fact_subscriptions

In [0]:
opportunity_df=spark.read.table(f"{SILVER_SCHEMA_PATH}.`silver_opportunity`")

## Creating the event surrogate key and dim_event

In [0]:
opportunity_df.createOrReplaceTempView("oppo_temp")
event_df=spark.sql("""
with cte as (
  select distinct close_status from oppo_temp
)
select close_status,row_number() over(order by close_status) as event_sk from cte 
""")
opportunity_df=opportunity_df.join(event_df,"close_status")

In [0]:
fact_subscription=opportunity_df.select("opportunity_id","customer_id","product_id","employee_id","start_date","end_date","revenue_amount","created_timestamp","Month","revenue_in_gpb","contract_months","mrr_in_gpb","event_sk")

# Creating Data Cube

In [0]:
subscription_analytics=fact_subscription\
    .join(customer_df,"customer_id","left")\
    .join(product_df,"product_id","left")\
    .join(event_df,"event_sk","left")\
    .select("product_id","customer_id","employee_id","start_date"
        ,"end_date","revenue_amount","Month","revenue_in_gpb"
        ,"mrr_in_gpb","customer_name","product_name","plan_name"
        ,"close_status")

# Saving Dataframe

### Applying SCD Type 1 for Fact Table

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(f"{GOLD_SCHEMA_PATH}.fact_subscription"):
    dt = DeltaTable.forName(spark, f"{GOLD_SCHEMA_PATH}.fact_subscription")
    dt.alias("target").merge(
        fact_subscription.alias("source"),
        "target.opportunity_id = source.opportunity_id" 
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    fact_subscription.write.format("delta").saveAsTable(f"{GOLD_SCHEMA_PATH}.fact_subscription")

### Overwriting other dimensional columns

In [0]:
product_df.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA_PATH}.`dim_product`")
event_df.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA_PATH}.`dim_event`")
subscription_analytics.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA_PATH}.`subscription_analytics`")

# Applying Z Ordering for Optimization

In [0]:
spark.sql(f"""
OPTIMIZE {GOLD_SCHEMA_PATH}.`fact_subscription`
ZORDER BY (customer_id, product_id)""")

In [0]:
spark.sql(f"""
OPTIMIZE {GOLD_SCHEMA_PATH}.`subscription_analytics`
ZORDER BY (customer_id, product_id)""")